# Week 84, Day 2: scikit-learn

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w084d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('flats.csv', 'w', encoding='utf-8') as f:
    f.write('size,rooms,area,garden,floor,rent\n61,2,South,yes,4,1150\n65,3,South,no,6,1140\n71,3,North,no,3,1110\n67,3,Riverside,no,3,1170\n90,3,Riverside,yes,3,1550\n62,2,Centre,yes,1,1370\n79,3,North,no,0,1140\n40,1,Centre,no,4,980\n43,2,Centre,no,4,1090\n108,4,Riverside,yes,2,1700\n65,3,North,yes,6,1210\n87,4,Riverside,no,5,1520\n105,4,Centre,yes,3,1860\n53,2,Centre,no,6,1160\n80,4,South,no,4,1240\n70,3,North,yes,0,1180\n61,3,North,no,0,980\n53,3,North,no,5,920\n90,4,South,yes,3,1380\n60,2,Riverside,yes,2,1240\n83,3,South,yes,3,1380\n58,2,Centre,yes,4,1270\n41,2,North,no,1,750\n103,4,Centre,no,6,1760\n39,2,North,yes,5,820\n70,3,Centre,no,4,1310\n60,2,North,yes,0,1060\n101,4,North,no,0,1430\n104,4,South,no,3,1560\n57,2,Centre,no,2,1170\n64,3,Riverside,no,1,1160\n85,3,South,no,2,1310\n69,3,North,no,3,1110\n85,4,Centre,yes,6,1570\n36,1,Centre,no,3,1040\n89,4,Centre,no,1,1550\n107,4,South,yes,5,1640\n93,4,South,no,2,1430\n63,3,North,yes,0,1110\n35,1,South,no,2,750\n66,3,Centre,no,0,1270\n63,2,Riverside,yes,2,1270\n51,2,North,yes,0,950\n46,2,Riverside,no,4,1040\n62,3,Riverside,no,2,1100\n60,3,South,no,0,970\n77,3,Centre,no,3,1400\n80,4,North,yes,6,1320\n')

**Predict before you run:** what will this print?



In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression

flats = pd.read_csv("flats.csv")
model = LinearRegression()
model.fit(flats[["size"]], flats["rent"])
print(round(model.coef_[0], 2), round(model.intercept_, 1))
print(round(model.predict(pd.DataFrame({"size": [70]}))[0]))
print(round(model.score(flats[["size"]], flats["rent"]), 3))

In [ ]:
import numpy as np
from sklearn.neighbors import KNeighborsClassifier

X = np.array([[1, 1], [1, 2], [2, 1], [6, 6], [6, 7], [7, 6]])
y = np.array(["red", "red", "red", "blue", "blue", "blue"])
knn = KNeighborsClassifier(n_neighbors=3).fit(X, y)
print(knn.classes_.tolist())
print(knn.predict([[1.5, 1.5], [6.5, 6.2]]).tolist())
print(knn.predict_proba([[4, 4]]).round(2).tolist())
print(knn.score(X, y))

In [ ]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

iris = load_iris(as_frame=True)
print(iris.frame.shape, iris.target_names.tolist())
print(iris.frame.head(2).values.tolist())
X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
print(X_train.shape, X_test.shape, np.bincount(y_test).tolist())

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, StandardScaler

train = np.array([[30.0], [60.0], [90.0]])
scaler = StandardScaler().fit(train)
print(scaler.mean_.tolist(), scaler.transform(train).round(2).ravel().tolist())
print(scaler.transform([[120.0]]).round(2).ravel().tolist())

areas = pd.DataFrame({"area": ["Centre", "North", "Centre", "South"]})
enc = OneHotEncoder(handle_unknown="ignore", sparse_output=False).fit(areas)
print(enc.get_feature_names_out().tolist())
print(enc.transform(pd.DataFrame({"area": ["North", "Riverside"]})).tolist())

In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

flats = pd.read_csv("flats.csv")
features = ["size", "rooms", "floor", "area", "garden"]
train, test = train_test_split(flats, test_size=0.25, random_state=0)
prep = ColumnTransformer([("num", "passthrough", ["size", "rooms", "floor"]),
                          ("cat", OneHotEncoder(drop="first"), ["area", "garden"])],
                         verbose_feature_names_out=False)
model = Pipeline([("prep", prep), ("model", LinearRegression())]).fit(train[features], train["rent"])
print(round(model.score(test[features], test["rent"]), 3))
names = model.named_steps["prep"].get_feature_names_out()
print({n: round(float(c), 1) for n, c in zip(names, model.named_steps["model"].coef_)})

In [ ]:
from sklearn.datasets import make_classification
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

X, y = make_classification(n_samples=300, n_features=6, n_informative=3, random_state=2)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)
for model in (DummyClassifier(strategy="most_frequent"), LogisticRegression(max_iter=1000),
              DecisionTreeClassifier(max_depth=3, random_state=0)):
    model.fit(X_train, y_train)
    print(type(model).__name__, round(model.score(X_test, y_test), 3))
tree = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_train, y_train)
print(tree.feature_importances_.round(2).tolist())

> **Key idea.** Every estimator is created with its hyperparameters, learns with `fit(X, y)` (`X` two-dimensional), predicts with `predict` (and `predict_proba` for classifiers), and scores with `score` (R² or accuracy); learned values end in `_`. `train_test_split(..., random_state=..., stratify=y)` splits reproducibly with balanced classes. Scale features for distance and gradient-based models with `StandardScaler`; one-hot encode categories with `OneHotEncoder` (`drop="first"` for a reference category, `handle_unknown="ignore"` for unseen ones). Fit preprocessing on training data only; a `Pipeline` (with a `ColumnTransformer` for mixed columns) makes that automatic and applies the same steps at prediction time. Compare models (linear, logistic, nearest neighbours, trees, forests) on the same test set, always against a `Dummy` baseline.

### Exercise: A first model

Write `rent_by_size(df)`: fit a `LinearRegression` predicting `rent` from `size` alone, and return `(model, slope, intercept)` with both numbers as floats rounded to 2 places. Then `predict_rents(model, sizes)`: the predicted rents for a list of sizes, as a list of whole numbers (rounded).

In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression

def rent_by_size(df):
    ___

def predict_rents(model, sizes):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import pandas as pd\n\ndef test_first():\n    \"fit, attributes, predict\"\n    df = pd.read_csv(\"flats.csv\")\n    model, slope, intercept = rent_by_size(df)\n    assert (slope, intercept) == (11.34, 448.2) and isinstance(slope, float)\n    assert predict_rents(model, [70, 30]) == [1242, 788]\n    assert round(model.score(df[[\"size\"]], df[\"rent\"]), 2) == 0.8")

### Exercise: Classifying irises

Write `iris_split(seed)`: load the iris data as arrays and split it 75/25 with `random_state=seed`, **stratified** by species; return the four parts. Then `knn_accuracy(k, seed)`: the test accuracy of a `KNeighborsClassifier` with `k` neighbours on that split, rounded to 3 places. And `per_class_test_counts(seed)`: how many test flowers of each species, as a list.

In [ ]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

def iris_split(seed):
    ___

def knn_accuracy(k, seed):
    ___

def per_class_test_counts(seed):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_iris():\n    \"stratified and accurate\"\n    X_train, X_test, y_train, y_test = iris_split(0)\n    assert X_train.shape == (112, 4) and X_test.shape == (38, 4)\n    for seed in (0, 1, 2):\n        assert sorted(per_class_test_counts(seed)) == [12, 13, 13], \"stratify keeps the species balanced\"\n    assert knn_accuracy(5, 0) >= 0.9 and knn_accuracy(1, 0) >= 0.9\n    assert knn_accuracy(5, 0) == knn_accuracy(5, 0), \"reproducible\"")

### Exercise: Why scaling matters

Write `make_data()`: with `rng = np.random.RandomState(0)`, draw 200 values `signal = rng.normal(0, 1, 200)`, then labels `y = (signal + rng.normal(0, 0.3, 200) > 0).astype(int)`, then `noise = rng.normal(0, 1000, 200)` (in that order), and return `X` (columns `signal` and `noise`) and `y`. Then `compare_scaling(X, y)`: split 70/30 with `random_state=1`, and return the test accuracies (3 places) of `KNeighborsClassifier(5)` without scaling and of a pipeline that scales first, as `(raw, scaled)`.

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def make_data():
    ___

def compare_scaling(X, y):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import numpy as np\n\ndef test_scaling():\n    \"the big feature drowns the useful one\"\n    X, y = make_data()\n    assert X.shape == (200, 2) and abs(X[:, 1].std() / X[:, 0].std()) > 300\n    raw, scaled = compare_scaling(X, y)\n    assert raw == 0.517 and scaled == 0.983")

### Exercise: One-hot encoding

Write `encode(train, test, column)`: fit a `OneHotEncoder` (with `handle_unknown="ignore"` and `sparse_output=False`) on the training DataFrame's `column`, and return `(names, train_encoded, test_encoded)`: the list of output column names and the two encoded arrays as lists of lists of whole numbers. A category that appears only in `test` becomes all zeros.

In [ ]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

def encode(train, test, column):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import pandas as pd\n\ndef test_encode():\n    \"one column per category\"\n    train = pd.DataFrame({\"area\": [\"Centre\", \"North\", \"Centre\"]})\n    test = pd.DataFrame({\"area\": [\"North\", \"Riverside\"]})\n    names, tr, te = encode(train, test, \"area\")\n    assert names == [\"area_Centre\", \"area_North\"]\n    assert tr == [[1, 0], [0, 1], [1, 0]] and te == [[0, 1], [0, 0]]")

### Exercise: A rent pipeline

Write `rent_pipeline()`: an unfitted `Pipeline` with a `ColumnTransformer` step named `"prep"` (scale `size`, `rooms` and `floor` with `StandardScaler`; one-hot encode `area` and `garden` with `handle_unknown="ignore"`) and a `LinearRegression` step named `"model"`. Then `train_and_score(df)`: split the flats 75/25 with `random_state=0`, fit a new pipeline on the training part, and return `(pipeline, r2)` with the test R² rounded to 3 places. And `predict_one(pipeline, flat)`: the predicted rent, rounded to a whole number, for a dict describing one flat.

In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

NUMERIC = ["size", "rooms", "floor"]
CATEGORIES = ["area", "garden"]

def rent_pipeline():
    ___

def train_and_score(df):
    ___

def predict_one(pipeline, flat):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import pandas as pd\n\ndef test_pipeline():\n    \"preprocess and predict in one object\"\n    p = rent_pipeline()\n    assert list(p.named_steps) == [\"prep\", \"model\"]\n    pipe, r2 = train_and_score(pd.read_csv(\"flats.csv\"))\n    assert r2 == 0.938\n    assert predict_one(pipe, {\"size\": 70, \"rooms\": 3, \"floor\": 2, \"area\": \"Centre\", \"garden\": \"yes\"}) == 1412\n    assert isinstance(predict_one(pipe, {\"size\": 50, \"rooms\": 2, \"floor\": 1, \"area\": \"Docks\", \"garden\": \"no\"}), int), \"unseen areas don't crash\"")

### Exercise: Against the baseline

Write `compare_models(X_train, y_train, X_test, y_test)`: fit a `DummyClassifier(strategy="most_frequent")`, `LogisticRegression(max_iter=1000)`, `DecisionTreeClassifier(max_depth=3, random_state=0)` and `KNeighborsClassifier(5)`, and return a dict from the names `"baseline"`, `"logistic"`, `"tree"` and `"knn"` to test accuracy, rounded to 3 places. Then `worth_using(scores, margin=0.05)`: the sorted names of models that beat the baseline by more than `margin`.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

def compare_models(X_train, y_train, X_test, y_test):
    ___

def worth_using(scores, margin=0.05):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from sklearn.datasets import make_classification\nfrom sklearn.model_selection import train_test_split\n\ndef test_compare():\n    \"same test set, a baseline included\"\n    X, y = make_classification(n_samples=300, n_features=6, n_informative=3, random_state=2)\n    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)\n    scores = compare_models(X_train, y_train, X_test, y_test)\n    assert set(scores) == {\"baseline\", \"logistic\", \"tree\", \"knn\"}\n    assert scores[\"baseline\"] == 0.489 and abs(scores[\"tree\"] - 0.778) < 0.03\n    assert abs(scores[\"logistic\"] - 0.889) < 0.02 and abs(scores[\"knn\"] - 0.889) < 0.02\n    assert worth_using(scores) == [\"knn\", \"logistic\", \"tree\"]\n    assert worth_using({\"baseline\": 0.9, \"a\": 0.92, \"b\": 0.97}) == [\"b\"]")

### Bug hunt: A scaler that peeked

The scaler is fitted again on the test data, so the test set is scaled with its **own** mean and the result is always centred on zero: `test mean after scaling: 0.0`. Scale the test data with the scaler fitted on the training data, so it prints `test mean after scaling: 2.68`.

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler

train = np.array([[1.0], [2.0], [3.0], [4.0]])
test = np.array([[5.0], [6.0]])
scaler = StandardScaler().fit(train)
scaled_test = StandardScaler().fit_transform(test)
print("test mean after scaling:", round(float(scaled_test.mean()), 2))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"fit on training data only\"\n    expect_output(\"test mean after scaling: 2.68\")")

### Bug hunt: Expected a 2D array

`fit` needs the features as a table (rows and columns), but `sizes` is a flat list, so this crashes with `ValueError: Expected a 2D array`. Pass the features in two dimensions so it prints `prediction: 1000`.

In [ ]:
from sklearn.linear_model import LinearRegression

sizes = [40, 60, 80]
rents = [800, 1000, 1200]
model = LinearRegression().fit(sizes, rents)
print("prediction:", round(float(model.predict([[60]])[0])))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"rows and columns\"\n    expect_output(\"prediction: 1000\")")

### Bug hunt: A different test set every time

Without `random_state`, every run shuffles differently, so scores can't be compared or reproduced. Make the split repeatable with `random_state=0` so it always prints `test rows: [2, 4, 8]`.

In [ ]:
from sklearn.model_selection import train_test_split

rows = list(range(10))
train, test = train_test_split(rows, test_size=3)
print("test rows:", sorted(test))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"reproducible splits\"\n    expect_output(\"test rows: [2, 4, 8]\")\n    expect_output(\"test rows: [2, 4, 8]\")")

### Puzzle: Scale, then classify

Arrange the lines to split the iris data, build a pipeline that scales then classifies, fit it on the training data and print its test accuracy. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the method that learns parameters from data
q = ___   # "accuracy" or "R²": what score() gives for a classifier
r = ___   # how many columns OneHotEncoder() makes from a column with 4 categories
s = ___   # the same with drop="first"
t = ___   # True or False: a fitted model's learned values have names ending in "_"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \"fit\" and q == \"accuracy\" and r == 4, \"p, q and r\"\n    assert s == 3 and t is True, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: an explainable rent model

Write `rent_model(df, seed=0)` for the flats. Split 75/25 with `random_state=seed`. Build a pipeline with a `ColumnTransformer` named `"prep"` that passes `size`, `rooms` and `floor` through unchanged and one-hot encodes `area` and `garden` with `drop="first"`, with `verbose_feature_names_out=False`, followed by a `LinearRegression` named `"model"`; fit it on the training part. Return `(pipeline, report)`, where the report has: `r2` (test, 3 places), `mae` (test mean absolute error, 1 place), `baseline_mae` (the MAE of always predicting the **training** mean rent, 1 place), `coefficients` (a dict from each feature name, such as `"size"` or `"area_North"`, to its coefficient, 1 place), `reference` (the dict of reference categories that were dropped, like `{"area": "Centre", "garden": "no"}`), and `biggest` (the name of the feature with the largest coefficient in absolute size). Then `explain(report, feature)`: a sentence: `"size: +12.2 per unit"` for a numeric feature and `"area_North: -237.4 compared with Centre"` for an encoded one.

In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

NUMERIC = ["size", "rooms", "floor"]
CATEGORIES = ["area", "garden"]

def rent_model(df, seed=0):
    ___

def explain(report, feature):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import pandas as pd\n\ndef test_report():\n    \"accuracy and baseline\"\n    pipe, r = rent_model(pd.read_csv(\"flats.csv\"))\n    assert r[\"r2\"] == 0.938 and r[\"mae\"] == 34.5 and r[\"baseline_mae\"] == 153.1\n    assert r[\"reference\"] == {\"area\": \"Centre\", \"garden\": \"no\"}\n    assert r[\"coefficients\"] == {\"size\": 12.2, \"rooms\": -31.6, \"floor\": 8.2, \"area_North\": -237.4,\n                                 \"area_Riverside\": -104.8, \"area_South\": -204.4, \"garden_yes\": 79.2}\n    assert r[\"biggest\"] == \"area_North\"\n\ndef test_explain():\n    \"sentences people can read\"\n    pipe, r = rent_model(pd.read_csv(\"flats.csv\"))\n    assert explain(r, \"size\") == \"size: +12.2 per unit\"\n    assert explain(r, \"area_North\") == \"area_North: -237.4 compared with Centre\"\n    assert explain(r, \"garden_yes\") == \"garden_yes: +79.2 compared with no\"\n    other = rent_model(pd.read_csv(\"flats.csv\"), seed=3)[1]\n    assert other[\"r2\"] > 0.85 and other[\"mae\"] < other[\"baseline_mae\"] / 2")

## Warm-up

Three lines replace yesterday's least-squares formulas: create the model, `fit` it to features and labels, and `predict`. The fitted parameters appear as attributes ending in an underscore (`coef_`, `intercept_`), and `score` gives R² for a regression. The features are a **table** (`flats[["size"]]`, double brackets), even with one column.

## The estimator pattern

Every scikit-learn model follows the same steps:

1. **Create** it, choosing its settings (**hyperparameters**): `KNeighborsClassifier(n_neighbors=5)`, `DecisionTreeClassifier(max_depth=3)`. Creating it learns nothing.
2. **`fit(X, y)`** learns parameters from training data. `X` is two-dimensional (rows are examples, columns are features: a DataFrame or a 2D array); `y` is one-dimensional. `fit` returns the model itself, so `model = Model().fit(X, y)` works.
3. **`predict(X)`** gives predictions for new rows; classifiers also have **`predict_proba(X)`**, the probability of each class.
4. **`score(X, y)`** gives a standard measure: R² for regressors, accuracy for classifiers.

What it learned is stored in attributes ending in `_`: `coef_`, `intercept_`, `classes_`, `feature_importances_`. Preprocessing objects (**transformers**) follow the same idea with `fit` and `transform`.

That's yesterday's k-nearest neighbours: same idea, one line.

## Data sets and splits

scikit-learn ships a few small classic data sets, such as `load_iris()` (150 iris flowers, four measurements each, three species), and can make synthetic ones (`make_classification`, `make_regression`) for experiments. Each returns `X` and `y` (`return_X_y=True`) or a bundle with names and a description (`as_frame=True` gives pandas tables).

**`train_test_split(X, y, test_size=0.25, random_state=0)`** shuffles and splits in one call, returning `X_train, X_test, y_train, y_test`. Set `random_state` so the split is the same every run (otherwise each run gets different test data and different scores). For classification, **`stratify=y`** keeps the class proportions the same in both parts, which matters when one class is rare.

## Preprocessing

Most models need numbers, on comparable scales.

- **Scaling.** Distance-based models (k-nearest neighbours) and those trained by gradient descent are thrown off when one feature is measured in thousands and another in units: the big one dominates every distance. **`StandardScaler`** turns each feature into z-scores (mean 0, standard deviation 1). Linear regression and trees don't need it.
- **Categories.** A column like `area` can't go into arithmetic as text, and numbering the areas 0, 1, 2, 3 would invent an order. **`OneHotEncoder`** makes one 0/1 column per category (`area_Centre`, `area_North`, ...). With `drop="first"`, one category becomes the reference the others are compared with; with `handle_unknown="ignore"`, a category never seen in training becomes all zeros instead of an error.

The rule that matters most: **fit preprocessing on the training data only**, then use those fitted numbers to transform both training and test data (`scaler.fit(X_train)`, then `scaler.transform(X_test)`). Fitting on all the data lets information about the test set **leak** into training, and the test score becomes optimistic.

The new flat of 120 m² is scaled using the **training** mean and spread; Riverside, never seen in training, encodes as all zeros.

## Pipelines

Doing the steps by hand is how leaks happen: a scaler fitted on the wrong data, an encoder forgotten at prediction time. A **`Pipeline`** chains transformers and a model into one estimator: `fit` fits each step on the training data in turn, and `predict` sends new data through the same fitted steps. **`ColumnTransformer`** applies different preprocessing to different columns (scale the numbers, encode the categories) and joins the results.

```python
prep = ColumnTransformer([("num", StandardScaler(), ["size", "rooms", "floor"]),
                          ("cat", OneHotEncoder(handle_unknown="ignore"), ["area", "garden"])])
model = Pipeline([("prep", prep), ("model", LinearRegression())])
model.fit(train_df, train_df["rent"])      # extra columns are dropped by the ColumnTransformer
model.predict(new_flats_df)
```

`make_pipeline(StandardScaler(), KNeighborsClassifier())` is a shorter way to write one, naming the steps for you. `model.named_steps["model"]` reaches a fitted step.

Read the coefficients: each square metre adds about £12 a month, a garden about £79, and the North costs about £237 less than the Centre (the dropped reference). The data was in fact made with £11, £70 and £250: the model recovered the pattern from 36 noisy examples.

## A tour of models

- **`LinearRegression`**: a weighted sum of the features; fast, easy to explain, assumes effects add up.
- **`LogisticRegression`**: despite the name, a **classifier**: a weighted sum turned into a probability; also easy to explain. Give it `max_iter=1000` if it warns that it hasn't converged.
- **`KNeighborsClassifier`** and `KNeighborsRegressor`: yesterday's idea; need scaling.
- **`DecisionTreeClassifier`** and `Regressor`: a flowchart of yes/no questions on features, learned from the data; `max_depth` limits how flexible it is (deep trees overfit); `feature_importances_` shows which features it used.
- **`RandomForestClassifier`**: many trees on random samples, voting; often strong with little tuning.
- **`DummyClassifier(strategy="most_frequent")`** and **`DummyRegressor(strategy="mean")`**: the baselines, as estimators. Always include one.

## Practice

## Quiz

## Daily challenge

## Before you go

Take the rent pipeline and try two changes, one at a time: replace `LinearRegression` with `DecisionTreeRegressor(max_depth=3)`, then with `KNeighborsRegressor(5)` (with and without scaling). Write down each test R² and MAE next to the baseline. Which would you trust, and which could you explain to a tenant? Tomorrow is **evaluating models**: what accuracy hides, and how to measure honestly.